# tinyaya-idk on Google Colab

Runtime: **Runtime > Change runtime type > T4 GPU**. Add your Hugging Face token as a Colab secret named `HF_TOKEN` (key icon in the left bar) and grant notebook access. Accept the TinyAya licence on its Hugging Face page first.

Repo: https://github.com/Eman2123/tinyaya-idk

In [ ]:
!nvidia-smi

In [ ]:
# Keep results on Google Drive so a disconnect does not lose them
from google.colab import drive, userdata
import os
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
DRIVE_OUT = "/content/drive/MyDrive/tinyaya-idk-outputs"
os.makedirs(DRIVE_OUT, exist_ok=True)

if not os.path.exists("/content/tinyaya-idk"):
    !git clone https://github.com/Eman2123/tinyaya-idk /content/tinyaya-idk
%cd /content/tinyaya-idk
!git pull -q
if not os.path.islink("outputs"):
    !ln -s {DRIVE_OUT} outputs
!pip install -q peft trl bitsandbytes datasets accelerate pyyaml scikit-learn pandas tqdm

In [ ]:
# ---- settings: edit these ----
MODEL_ID = "CohereLabs/tiny-aya-global"   # exact TinyAya Hugging Face ID (confirm on the Cohere Labs page)
ANCHOR   = "en"
TARGETS  = ["es", "ar", "ms"]                      # MKQA has ms but no id/sw; confirm TinyAya supports each
N_QUESTIONS = 1500                                 # drop to 800 if the GPU is too slow
LOAD_4BIT = True                                   # True for a T4 (16 GB)
SEED = 1
LANGS = [ANCHOR] + TARGETS
LANGS_STR = " ".join(LANGS)
TAG = "-".join(LANGS)

In [ ]:
import yaml
cfg = yaml.safe_load(open("configs/config.yaml"))
cfg["model"]["name"] = MODEL_ID
cfg["model"]["load_in_4bit"] = LOAD_4BIT
cfg["languages"] = {"anchor": ANCHOR, "targets": TARGETS}
yaml.safe_dump(cfg, open("configs/config.yaml", "w"), allow_unicode=True, sort_keys=False)
print("model:", cfg["model"]["name"], "| langs:", LANGS)

## 1. Smoke test
Run this first. If it fails, fix the model ID / access / memory before anything else.

In [ ]:
# Smoke test: load the model and answer 10 questions in the anchor language (fix problems here first)
!python src/prepare_data.py --langs {LANGS_STR} --n {N_QUESTIONS}
!python src/sample.py --lang {ANCHOR} --limit 10
!python src/labels.py --lang {ANCHOR}

## 2. Sampling and labels
Note: the smoke test wrote a 10-question file for the anchor. The cell below re-samples the anchor fully (it does not skip the anchor on purpose). If a later language fails, delete `outputs/samples/<lang>.jsonl` and `outputs/hidden/<lang>.npz` and re-run.

In [ ]:
# Full sampling (5 samples + greedy + hidden states). Skips languages that are already done,
# so you can re-run this cell after a disconnect.
import os
for L in LANGS:
    if os.path.exists(f"outputs/hidden/{L}.npz") and L != ANCHOR:
        print("skip", L); continue
    !python src/sample.py --lang {L}
    !python src/labels.py --lang {L}

## 3. Label quality (RQ1) and training sets

In [ ]:
# RQ1: label quality (consistency + transferred probe vs gold). Look at this table before any training.
!python src/probe.py --anchor {ANCHOR} --targets {" ".join(TARGETS)}
!python src/build_datasets.py --langs {LANGS_STR}

## 4. Baseline + R2/R3/R4 training and evaluation
Time the first run before committing to all languages. If it is too slow: lower `N_QUESTIONS`, drop a language, or keep `LOAD_4BIT = True`.

In [ ]:
import os
# Baseline first: over-refusal and abstention metrics need the base model's predictions.
if not os.path.exists("outputs/results/eval_R0.csv"):
    !python src/evaluate.py --name R0 --langs {LANGS_STR}
    !python src/sanity_checks.py --name R0

for R in ["R2", "R3", "R4"]:
    name = f"{R}_{TAG}_s{SEED}"
    if not os.path.exists(f"outputs/adapters/{name}"):
        !python src/train_sft.py --recipe {R} --langs {LANGS_STR} --seed {SEED}
    if not os.path.exists(f"outputs/results/eval_{name}.csv"):
        !python src/evaluate.py --name {name} --adapter outputs/adapters/{name} --langs {LANGS_STR}
        !python src/sanity_checks.py --name {name} --adapter outputs/adapters/{name}

## 5. Transfer experiments

In [ ]:
# Cross-lingual transfer: train on the anchor only, test on every language.
for R in ["R2", "R3", "R4"]:
    name = f"{R}_{ANCHOR}_s{SEED}"
    if not os.path.exists(f"outputs/adapters/{name}"):
        !python src/train_sft.py --recipe {R} --langs {ANCHOR} --seed {SEED}
    if not os.path.exists(f"outputs/results/eval_{name}.csv"):
        !python src/evaluate.py --name {name} --adapter outputs/adapters/{name} --langs {LANGS_STR}

## 6. Tables

In [ ]:
!python src/stats.py summary
!python src/stats.py bootstrap --a R2 --b R3 --langs {LANGS_STR}
!python src/stats.py bootstrap --a R3 --b R4 --langs {LANGS_STR}
!python src/stats.py transfer --recipe R3

## 7. Save

In [ ]:
# Results already live on Drive (outputs -> Drive). Optional: also download a zip.
!zip -qr /content/tinyaya_results.zip outputs/results
from google.colab import files
files.download("/content/tinyaya_results.zip")